# Peer Benchmarking — Z-Score & Percentile Ranking
## Greek Systemic Banks (2022–2024)

Z-score and percentile-rank each bank against the 4-bank sector peer group per year.
This is the framework used by equity research analysts at bulge-bracket and regional banks.

**Methodology:**
- Z-score: `z = (x − μ) / σ` where μ and σ are the cross-sectional mean/std of 4 peers in a given year.
- Percentile rank: `percentile = (rank − 1) / (n − 1) × 100` (higher = better within peer group).
- For 'lower-is-better' metrics (C/I, L/D, Cost of Risk), signs are flipped so that percentile 100 always means best-in-class.

**KPIs ranked:** ROE, ROA, NIM, Cost-to-Income, Loan-to-Deposit, CET1.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
YEARS = [2022, 2023, 2024]

DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} rows.')

def hex_rgba(h, a=0.25):
    h = h.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{a})'


Loaded 12 rows.


In [2]:
# ── Define KPIs and their direction ──────────────────────────────────────────
KPI_META = {
    'roe':            ('ROE (%)',            'higher'),
    'roa':            ('ROA (%)',            'higher'),
    'nim':            ('NIM (%)',            'higher'),
    'cost_to_income': ('Cost/Income (%)',    'lower'),
    'loan_to_deposit':('Loan/Deposit (%)',   'lower'),
    'cet1':           ('CET1 (%)',           'higher'),
}
KPIS       = list(KPI_META.keys())
KPI_LABELS = [KPI_META[k][0] for k in KPIS]

df = kpis[['bank','year'] + KPIS].copy()

# ── Compute z-scores and percentile ranks per year ────────────────────────────
# Explicit year loop avoids pandas 3.x groupby-key-dropping behaviour.
z_cols, pct_cols = [], []
for kpi, (_, direction) in KPI_META.items():
    z_col, pct_col = f'z_{kpi}', f'pct_{kpi}'
    z_cols.append(z_col); pct_cols.append(pct_col)
    df[z_col] = 0.0; df[pct_col] = 0
    for year in YEARS:
        mask = df['year'] == year
        vals = df.loc[mask, kpi]
        mu, sigma = vals.mean(), vals.std(ddof=1)
        z = (vals - mu) / sigma if sigma > 0 else pd.Series([0.0] * mask.sum(), index=vals.index)
        if direction == 'lower':
            z = -z
        df.loc[mask, z_col]  = z.round(2)
        rank = z.rank(ascending=True)
        df.loc[mask, pct_col] = ((rank - 1) / (mask.sum() - 1) * 100).round(0).astype(int)

print('Z-scores and percentile ranks computed.')
print(df[['bank','year'] + z_cols].round(2).to_string(index=False))

Z-scores and percentile ranks computed.
        bank  year  z_roe  z_roa  z_nim  z_cost_to_income  z_loan_to_deposit  z_cet1
  Alpha Bank  2022  -1.35  -1.39  -0.84             -1.40              -1.17   -0.62
  Alpha Bank  2023  -1.12  -1.07  -1.25             -1.36              -1.03   -0.57
  Alpha Bank  2024  -1.32  -1.42  -0.89             -1.06              -1.48   -0.44
    Eurobank  2022   0.95   0.91   1.44              0.97              -0.49    0.56
    Eurobank  2023   0.96   0.78   0.19              1.04              -0.65    0.46
    Eurobank  2024   1.11   0.49  -0.44              0.71               0.40    0.48
         NBG  2022   0.52   0.51  -0.19              0.11               0.80    1.10
         NBG  2023   0.72   0.92   1.18              0.06               1.06    1.17
         NBG  2024   0.22   0.87   1.41             -0.64               0.70    1.11
Piraeus Bank  2022  -0.11  -0.03  -0.41              0.32               0.86   -1.05
Piraeus Bank  2023  -0.55

In [3]:
# ── Chart 1: Radar / Spider Charts per Bank (2024 focus) ─────────────────────
# Percentile rank (0-100) on each KPI, plotted on radar. 100 = best in peer group.

df24 = df[df['year'] == 2024].set_index('bank')

fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    specs=[[{'type': 'polar'}]*2]*2,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

theta = KPI_LABELS + [KPI_LABELS[0]]  # close the polygon

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    pct_vals = [df24.loc[bank, f'pct_{k}'] for k in KPIS]
    r = pct_vals + [pct_vals[0]]  # close

    fig1.add_trace(
        go.Scatterpolar(
            r=r,
            theta=theta,
            fill='toself',
            fillcolor=hex_rgba(COLORS[bank], 0.25),
            line=dict(color=COLORS[bank], width=2.5),
            name=bank,
            showlegend=False,
        ),
        row=row + 1, col=col + 1,
    )

    # Annotate each point
    for ri, (kpi_label, val) in enumerate(zip(KPI_LABELS, pct_vals)):
        fig1.add_trace(
            go.Scatterpolar(
                r=[val],
                theta=[kpi_label],
                mode='markers+text',
                marker=dict(color=COLORS[bank], size=8),
                text=[f'{val}%'],
                textposition='top center',
                textfont=dict(size=9),
                showlegend=False,
            ),
            row=row + 1, col=col + 1,
        )

fig1.update_polars(
    radialaxis=dict(
        range=[0, 100], tickvals=[0, 33, 67, 100],
        ticktext=['0', '33', '67', '100'],
        gridcolor='rgba(255,255,255,0.15)',
        linecolor='rgba(255,255,255,0.15)',
    ),
    angularaxis=dict(gridcolor='rgba(255,255,255,0.15)'),
    bgcolor='#0f1117',
)
fig1.update_layout(
    title_text='<b>Peer Benchmarking Radar — 2024</b> | Percentile rank within sector (100 = best in class)',
    title_font_size=16,
    template='plotly_dark',
    height=640,
    paper_bgcolor='#0f1117',
)
fig1.show()

In [4]:
# ── Chart 2: Ranked Bar Charts per KPI (all years) ───────────────────────────
# For each KPI, show all 4 banks as grouped bars across years.

fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=KPI_LABELS,
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

bar_opacity = 0.85
for ki, (kpi, (label, direction)) in enumerate(KPI_META.items()):
    r, c = positions[ki]
    for bank in BANKS:
        bdf = df[df['bank'] == bank].sort_values('year')
        fig2.add_trace(
            go.Bar(
                x=[f"{b}\n{y}" for b, y in zip([bank]*3, YEARS)],
                y=bdf[kpi].values,
                name=bank,
                marker_color=COLORS[bank],
                opacity=bar_opacity,
                showlegend=(ki == 0),
                legendgroup=bank,
            ),
            row=r, col=c,
        )

fig2.update_layout(
    title_text='<b>KPI Comparison — All Banks & Years</b>',
    title_font_size=16,
    template='plotly_dark',
    height=560,
    barmode='group',
    legend=dict(orientation='h', y=-0.10, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.show()

In [5]:
# ── Chart 3: Composite Peer Score Heatmap ─────────────────────────────────────
# Average z-score across all KPIs per bank-year; higher = stronger overall peer position.

df['composite_z'] = df[z_cols].mean(axis=1).round(2)

pivot = df.pivot_table(index='bank', columns='year', values='composite_z').reindex(BANKS)

fig3 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(y) for y in YEARS],
    y=BANKS,
    text=[[f'{v:+.2f}' for v in row] for row in pivot.values],
    texttemplate='%{text}',
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title='Avg Z-score', tickformat='+.1f'),
))
fig3.update_layout(
    title_text='<b>Composite Peer Z-Score Heatmap</b> | Average across 6 KPIs | Positive = above sector mean',
    title_font_size=15,
    template='plotly_dark',
    height=300,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    xaxis_title='Year',
)
fig3.show()

print('\n── 2024 Composite Z-Score Ranking ──')
df24_z = df[df['year']==2024].set_index('bank')['composite_z'].sort_values(ascending=False)
for rank, (bank, z) in enumerate(df24_z.items(), 1):
    print(f'  #{rank} {bank}: {z:+.2f}')


── 2024 Composite Z-Score Ranking ──
  #1 NBG: +0.61
  #2 Eurobank: +0.46
  #3 Piraeus Bank: +0.03
  #4 Alpha Bank: -1.10


In [6]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows.'

# Per-year z-scores must have zero mean across the 4 banks
for year in YEARS:
    yr_df = df[df['year'] == year]
    for z_col in z_cols:
        mean_z = yr_df[z_col].mean()
        assert abs(mean_z) < 0.01, f'Non-zero mean z-score in {year} for {z_col}: {mean_z:.4f}'

# Percentile ranks must cover {0, 33, 67, 100} for 4 banks
expected_pcts = {0, 33, 67, 100}
for year in YEARS:
    yr_df = df[df['year'] == year]
    for pct_col in pct_cols:
        actual = set(yr_df[pct_col].values.tolist())
        assert actual == expected_pcts, f'{pct_col} {year}: expected {expected_pcts}, got {actual}'

print('✅ All checks passed.')
print(f'   Z-score means are zero for all KPIs in all years (confirms correct normalisation).')
print(f'   Percentile ranks cover {{0,33,67,100}} for all KPIs in all years.')

✅ All checks passed.
   Z-score means are zero for all KPIs in all years (confirms correct normalisation).
   Percentile ranks cover {0,33,67,100} for all KPIs in all years.
